# Build and Train the `CNN Model` using `PyTorch`

## import libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

---------

##  Data Preprocessing

In [10]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize pixel values
])

train_path = r"C:\Users\bhaut\AI Tensorflow intellipaat\datasets\train"
test_path = r"C:\Users\bhaut\AI Tensorflow intellipaat\datasets\test"

# Load the FER2013 dataset
train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
val_dataset = datasets.ImageFolder(root=test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Number of classes (e.g., Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral)
num_classes = len(train_dataset.classes)
print("Classes:", train_dataset.classes)

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


------------

## Create a simple CNN architecture

In [11]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super(EmotionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1 channel (grayscale)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),  # Adjust based on image size after conv layers
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # Output layer for classification
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Instantiate the model
model = EmotionCNN(num_classes=num_classes)
print(model)

EmotionCNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=7, bias=True)
  )
)


---------

## Define Loss Function and Optimizer

In [12]:
criterion = nn.CrossEntropyLoss()  # Cross-entropy loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

---------------

## Train the CNN model

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(50):
    model.train()
    train_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{50}], Loss: {train_loss/len(train_loader):.4f}")

Epoch [1/50], Loss: 1.3709
Epoch [2/50], Loss: 1.2442
Epoch [3/50], Loss: 1.1513
Epoch [4/50], Loss: 1.0710
Epoch [5/50], Loss: 1.0000
Epoch [6/50], Loss: 0.9160
Epoch [7/50], Loss: 0.8533
Epoch [8/50], Loss: 0.7754
Epoch [9/50], Loss: 0.7152
Epoch [10/50], Loss: 0.6443
Epoch [11/50], Loss: 0.5931
Epoch [12/50], Loss: 0.5373
Epoch [13/50], Loss: 0.4996
Epoch [14/50], Loss: 0.4609
Epoch [15/50], Loss: 0.4214
Epoch [16/50], Loss: 0.4008
Epoch [17/50], Loss: 0.3703
Epoch [18/50], Loss: 0.3476
Epoch [19/50], Loss: 0.3362
Epoch [20/50], Loss: 0.3164
Epoch [21/50], Loss: 0.3008
Epoch [22/50], Loss: 0.2893
Epoch [23/50], Loss: 0.2851
Epoch [24/50], Loss: 0.2752
Epoch [25/50], Loss: 0.2688
Epoch [26/50], Loss: 0.2572
Epoch [27/50], Loss: 0.2528
Epoch [28/50], Loss: 0.2332
Epoch [29/50], Loss: 0.2437
Epoch [30/50], Loss: 0.2205
Epoch [31/50], Loss: 0.2253
Epoch [32/50], Loss: 0.2218
Epoch [33/50], Loss: 0.2189
Epoch [34/50], Loss: 0.2134
Epoch [35/50], Loss: 0.2048
Epoch [36/50], Loss: 0.2074
E

-------------

## Evaluate the Model

In [16]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 58.36%


----------

## Save the trained Model

In [17]:
torch.save(model.state_dict(), "emotion_model.pth")
print("Model saved successfully!")

Model saved successfully!


-----